In [80]:
import os.path
import gc
import pandas as pd
import pickle as pkl
import numpy as np
import random
from tqdm import tqdm
import json
from sklearn import metrics
from utils import get_texts_for_ptype
from sklearn import metrics
from custom_models.custom_model import BatchVectorizerClassifier, BatchVectorizerClassifier
from ray import tune
from scipy.optimize import minimize_scalar
# from classificationconfidenceintervals import ClassificationConfidenceIntervals

In [2]:
neg_to_pos_ratio = 1

In [3]:
results_file = "results_titles_abstracts_linear_features1000.json"
with open(results_file, "r") as f_read:
    results_list = json.load(f_read)

In [47]:
decision_fn_file = "results_decision_fn_linear_features1000.json"

In [48]:
with open(decision_fn_file, "r") as f_read:
    df_list = json.load(f_read)

In [6]:
def predict(df, db=0):
    return [int(x>=db) for x in df ]

In [7]:
neg_pos_ratio_dict = {
    "is_rct": 5585042/489295,
    "is_sr": 5659292/121063,
    "is_casecontrol": 5628219/65790,
    "is_cohort": 5632460/78688,
    "is_guideline": 5676411/33826
}

def calc_precision(recall, specificity, neg_pos_ratio):
    return recall / (recall + neg_pos_ratio * (1 - specificity) + 1e-15)

def calc_f1(recall, precision):
    return 2*recall*precision/(recall+precision + 1e-15)

In [8]:
def find_db(db):
    y_pred_test = predict(df, db)
    recall_test = metrics.recall_score(y_test, y_pred_test, pos_label=1)
    return abs(recall_test - 0.985)

In [10]:
ptype = "is_rct"
X_train, X_val, X_test, y_train, y_val, y_test = get_texts_for_ptype(ptype, inputs="titles_abstracts", neg_to_pos_ratio=neg_to_pos_ratio)    

19it [00:23,  1.36s/it]/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
51it [01:09,  1.33s/it]/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
86it [01:56,  1.36s/it]
57it [01:30,  1.59s/it]


In [83]:
model_type = "perceptron"
for results_json, df in zip(results_list, df_list):
    
    if results_json["ptype"] == ptype and results_json["model_type"] == model_type:
        print(results_json["model_type"] == model_type)
        break

In [84]:
def get_scores(predictions):
    return [np.exp(p[1])/(np.exp(p[0]) + np.exp(p[1])) for p in predictions]

In [85]:
from custom_models.custom_model import BatchVectorizerXGBoost

results_file = "results_titles_abstracts_linear_features1000.json"
    
with open(results_file, "r") as f:
    results_list = json.load(f)
    
for results_json in results_list:
    
    if results_json["ptype"] == ptype and results_json["model_type"] == model_type:
        break
bvc = BatchVectorizerClassifier(
                    loss=results_json["model_type"],
                    penalty=results_json["best_penalty"],
                    alpha=results_json["best_alpha"],
                    l1_ratio=0.15,
                    fit_intercept=True,
                    max_iter=1000,
                    tol=0.001,
                    shuffle=True,
                    verbose=0,
                    epsilon=0.1,
                    n_jobs=None,
                    random_state=None,
                    learning_rate="optimal",
                    eta0=0.0,
                    power_t=0.5,
                    early_stopping=False,
                    validation_fraction=0.1,
                    n_iter_no_change=5,
                    class_weight=None,
                    warm_start=False,
                    average=False,
        )
bvc.fit(X_train, y_train)
df = bvc.decision_function(X_test)


In [86]:
# scores = get_scores(predictions)
# df = scores

In [90]:
auc_roc = metrics.roc_auc_score(y_test, df)
# res = minimize_scalar(find_db)
# db = res["x"]
db = -346.37577
y_pred_test = predict(df, db)
recall_test = metrics.recall_score(y_test, y_pred_test, pos_label=1)
spec_test = metrics.recall_score(y_test, y_pred_test, pos_label=0)
precision_test = calc_precision(recall_test, spec_test, neg_pos_ratio_dict[ptype])
print("db", db)
print("recall", recall_test)
print("spec", spec_test)
print("precision_test", precision_test)

db -346.37577
recall 0.988
spec 0.19425
precision_test 0.0970034248542508


In [34]:
df

[-0.2848227807586776,
 -2.5046190464672913,
 -0.9604350904882175,
 1.747927553879168,
 -2.104795150784365,
 -0.30095009952169294,
 -1.0049489034533128,
 -0.013054458159048599,
 -1.1898265401790835,
 2.1179103059581497,
 2.99680465633814,
 1.352340224019771,
 0.7703141157747972,
 1.189487676093791,
 -2.445377972332984,
 -0.5631869278098136,
 0.3477592742698574,
 1.3996022748095043,
 0.6156563738680241,
 -0.8109087667457072,
 0.530358273587384,
 -0.6494822788231595,
 -1.1407801704073464,
 -0.6924946186063006,
 1.321577221717816,
 -1.7211816593078861,
 -0.6896924967125158,
 -0.8460873472145477,
 -1.747542855842146,
 -0.14760238609062348,
 1.0249139737353263,
 -0.4179094116044687,
 1.8899029049911813,
 -1.3879145723179003,
 -0.05884817661752284,
 -3.1867314267355766,
 2.8526172974760464,
 -0.900293867770477,
 1.159722970243899,
 0.6801285188481176,
 0.7353876869281795,
 -0.602819921985611,
 -0.7807567846142701,
 -0.6643539904025844,
 4.789448704547755,
 -1.7546510347574464,
 -1.35767448250

In [15]:
metrics_lst = []
for results_json, df in tqdm(zip(results_list, df_list)):
    ptype = results_json["ptype"]
    model_type = results_json["model_type"]
    X_train, X_val, X_test, y_train, y_val, y_test = get_texts_for_ptype(ptype, inputs="titles_abstracts", neg_to_pos_ratio=neg_to_pos_ratio)    
    auc_roc = metrics.roc_auc_score(y_test, df)
    
    res = minimize_scalar(find_db)
    db = res["x"]
    y_pred_test = predict(df, db)
    
    recall_test = metrics.recall_score(y_test, y_pred_test, pos_label=1)
    spec_test = metrics.recall_score(y_test, y_pred_test, pos_label=0)
    precision_test = calc_precision(recall_test, spec_test, neg_pos_ratio_dict[ptype])
    f1_test = calc_f1(recall_test, precision_test)
    
    metrics_dict = {
        "ptype": ptype,
        "model_type": model_type,
        "y_test": y_test,
        "db": db,
        "df": df,
        "auc_roc": auc_roc,
        "recall_test": recall_test,
        "spec_test": spec_test,
        "precision_test": precision_test,
        "f1_test": f1_test
    }
    metrics_lst.append(metrics_dict)
    with open("results_adjusted_metrics.json", "wb") as f:
        pkl.dump(metrics_lst, f)

0it [00:00, ?it/s]
0it [00:00, ?it/s]
1it [00:01,  1.32s/it]
2it [00:02,  1.36s/it]
3it [00:03,  1.30s/it]
4it [00:05,  1.29s/it]
5it [00:06,  1.32s/it]
6it [00:07,  1.32s/it]
7it [00:09,  1.34s/it]
8it [00:10,  1.32s/it]
9it [00:11,  1.32s/it]
10it [00:13,  1.34s/it]
11it [00:14,  1.30s/it]
12it [00:15,  1.29s/it]
13it [00:17,  1.31s/it]
14it [00:18,  1.33s/it]
15it [00:19,  1.36s/it]
16it [00:21,  1.37s/it]
17it [00:22,  1.33s/it]
18it [00:23,  1.33s/it]
19it [00:25,  1.37s/it]/home/k2141255/.local/lib/python3.8/site-packages/tqdm/std.py:1182: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:

20it [00:26,  1.37s/it]
21it [00:27,  1.34s/it]
22it [00:29,  1.36s/it]
23it [00:30,  1.34s/it]
24it [00:31,  1.31s/it]
25it [00:33,  1.31s/it]
26it [00:34,  1.33s/it]
27it [00:35,  1.32s/it]
28it [00:37,  1.31s/it]
29it [00:38,  1.28s/it]
30it [00:39,  1.29s/it]
31it [00:41,  1.33s/it]
32it [00:42,  1.35s/it]
33it [00:43

In [15]:
with open("results_adjusted_metrics_linear_classifiers.json", "rb") as f:
    metrics_lst = pkl.load(f)

In [18]:
for metric in metrics_lst:
    print(metric['ptype'], metric['model_type'], metric['db'])


is_rct hinge -1.584230192290271
is_rct log_loss -1.7706254025899768
is_rct modified_huber -0.6220639543776835
is_rct squared_hinge -0.5470705538661514
is_rct perceptron -346.37576703005806
is_sr hinge -0.7258719104752848
is_sr log_loss -1.618033952508024
is_sr modified_huber -0.4016100443280388
is_sr squared_hinge -0.40188485444388444
is_sr perceptron -1.5556670765341585
is_casecontrol hinge -1.4974858741308337
is_casecontrol log_loss -1.7368543822025762
is_casecontrol modified_huber -0.7728828919222953
is_casecontrol squared_hinge -0.6817098011407674
is_casecontrol perceptron -1.0296123032304467
is_cohort hinge -1.131541471964921
is_cohort log_loss -1.3820771039068571
is_cohort modified_huber -0.5827571626219634
is_cohort squared_hinge -1.384765089115187
is_cohort perceptron -4.236067960975427
is_guideline hinge -0.9691094686657524
is_guideline log_loss -1.1016297746807175
is_guideline modified_huber -0.6282272743293027
is_guideline squared_hinge -4.236067960975427
is_guideline percep

In [78]:
with open("results_adjusted_metrics_ensemble_text_node2vec_gbdt.json", "rb") as f:
    metrics_lst = pd.read_pickle(f)

In [79]:
for metric in metrics_lst:
    print(metric['ptype'], metric['db'], metric["recall_test"])

is_rct -1.6180339754913822 1.0
is_rct -1.6180339754913822 1.0
is_rct -1.6180339754913822 1.0
is_sr -1.6180339754913822 1.0
is_casecontrol -1.6180339754913822 1.0
is_cohort -1.6180339754913822 1.0
is_guideline -1.6180339754913822 1.0
